# space-awareness-ml: run everything

Runtime > Change runtime type > T4 GPU, then run all. Every stage skips itself if its output already exists, so after a disconnect just run all again. Results live on Google Drive in `MyDrive/space-awareness-ml/results`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from pathlib import Path
from google.colab import drive

REPO_URL = "https://github.com/samiurk70/space-awareness-ml"
REPO = Path("/content/space-awareness-ml")
DATA = Path("/content/data")

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/space-awareness-ml")
(DRIVE / "results").mkdir(parents=True, exist_ok=True)
(DRIVE / "runs").mkdir(parents=True, exist_ok=True)
(DRIVE / "cache").mkdir(parents=True, exist_ok=True)
DATA.mkdir(exist_ok=True)

In [ ]:
res = REPO / "results"
if REPO.exists():
    # put the tracked results/ back before pulling, otherwise git sees it as deleted and the pull fails
    if res.is_symlink():
        res.unlink()
    !git -C {REPO} checkout -- results
    !git -C {REPO} pull --ff-only
else:
    !git clone {REPO_URL} {REPO}

# results/ inside the repo points at Drive so every script writes straight to persistent storage
if res.exists() and not res.is_symlink():
    !cp -rn {res}/. {DRIVE}/results/ && rm -rf {res}
if not res.exists():
    res.symlink_to(DRIVE / "results")

%cd {REPO}
!pip install -q -r requirements.txt

In [ ]:
zip_path = DRIVE / "cache" / "DUST.zip"
if not zip_path.exists():
    !wget -q --show-progress -O {zip_path}.part "https://zenodo.org/records/20255672/files/DUST.zip?download=1" && mv {zip_path}.part {zip_path}
if not (DATA / "DUST" / "SET 1").exists():
    !unzip -q {zip_path} -d {DATA}
code_zip = DRIVE / "cache" / "DUST_code.zip"
if not code_zip.exists():
    !wget -q -O {code_zip} "https://zenodo.org/records/20255672/files/DUST%20code.zip?download=1"

print("DUST tree, directories to depth 3, with file counts:")
for d in sorted(p for p in (DATA / "DUST").glob("*/*/*") if p.is_dir()):
    print(f"{len(list(d.iterdir())):5d}  {d.relative_to(DATA)}")

In [ ]:
import pandas as pd
from huggingface_hub import snapshot_download

ROBO6 = DATA / "robo6"
robo6_tar = DRIVE / "cache" / "robo6_mag.tar"

def robo6_problems():
    bad = []
    for split, n in [("train", 5676), ("test", 1404)]:
        df = pd.read_csv(ROBO6 / f"{split}.csv")
        assert len(df) == n, (split, len(df))
        for path in df.mag.unique():
            f = ROBO6 / path.replace("_time.txt", "_mag.txt")
            if not f.exists() or sum(1 for _ in open(f)) != 10000:
                bad.append(f)
    return bad

if not (ROBO6 / "train.csv").exists() and robo6_tar.exists():
    !tar -xf {robo6_tar} -C {DATA}
if not (ROBO6 / "train.csv").exists():
    # Only the magnitude files are needed: every file is already on a 0.1 s grid of 10,000 points.
    snapshot_download("kyselica/RoBo6", repo_type="dataset", local_dir=ROBO6,
                      allow_patterns=["*.csv", "README.md", "data/*/*_mag.txt"])
bad = robo6_problems()
print("RoBo6 missing or truncated files:", len(bad), bad[:5])
assert not bad, "rerun this cell to fetch the missing files"
if not robo6_tar.exists():
    !tar -cf {robo6_tar} -C {DATA} --exclude=.cache robo6

## Project A: DUST prep

In [ ]:
A_DATA = DATA / "dust_yolo"
if not (A_DATA / "gt.json").exists():
    !python a_dust/prep.py --dust {DATA}/DUST --out {A_DATA} --results results
else:
    print("prep already done")

In [ ]:
# CHECKPOINT A1: copy the tree printed above, this JSON and the figure back for review
!python a_dust/figures.py --data {A_DATA} --only method
print(open("results/prep_stats.json").read())
from IPython.display import Image, display
display(Image("results/figures/a_method.png"))

## Project A: training

12 runs: V0 to V3 x seeds 0, 1, 2. Ordered so that if time runs out, all four variants at seed 0 and V0 and V3 at all seeds finish first. Each run resumes from its last checkpoint on Drive after a disconnect.

In [ ]:
A_RUNS = DRIVE / "runs" / "a"
ORDER = ([(v, 0) for v in ["V0", "V1", "V2", "V3"]]
         + [(v, s) for s in [1, 2] for v in ["V0", "V3"]]
         + [(v, s) for s in [1, 2] for v in ["V1", "V2"]])
for v, s in ORDER:
    print(f"===== {v} seed {s}")
    !python a_dust/train.py --data {A_DATA} --variant {v} --seed {s} --project {A_RUNS}

In [ ]:
for v, s in ORDER:
    run = A_RUNS / f"{v}_s{s}"
    state = "done" if (run / "DONE").exists() else ("partial" if (run / "weights" / "last.pt").exists() else "not started")
    print(f"{v} seed {s}: {state}")

## Project A: evaluation, tracking, figures

Works with whichever runs have finished, so it can be run early as a check and again once all 12 are done. Predictions are cached next to each run on Drive.

In [ ]:
!python a_dust/evaluate.py --data {A_DATA} --dust {DATA}/DUST --runs {A_RUNS} --results results
if (A_RUNS / "V3_s0" / "test_preds.json").exists():
    !python a_dust/track.py --data {A_DATA} --dust {DATA}/DUST --run {A_RUNS}/V3_s0 --results results
!python a_dust/figures.py --data {A_DATA} --results results --only all

In [ ]:
# CHECKPOINT A2: a_results.json and a_ablation.png
import json
from IPython.display import Image, display
res = json.load(open("results/a_results.json"))
for v, d in res["variants"].items():
    m = d["metrics"]
    print(v, "seeds", d["seeds"], "  ".join(f"{k} {m[k]['mean']:.3f}+-{m[k]['std']:.3f}" for k in ["P", "R", "F1", "AP", "mAP50_inflated"]))
if "tracking" in res:
    print("tracking", {k: round(v, 3) for k, v in res["tracking"]["per_session"]["OVERALL"].items()})
for f in ["a_ablation.png", "a_pr.png", "a_brightness.png", "a_speed_stars.png"]:
    display(Image(f"results/figures/{f}"))

## Project B

These cells are added next.

In [ ]:
# In Cursor: Colab sidebar > /content > right-click results.zip > Download, save it in the local repo root
!rm -f /content/results.zip && cd {DRIVE} && zip -qr /content/results.zip results && ls -lh /content/results.zip